<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/llm_graph_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. NetworkX를 활용한 메모리 기반 지식 그래프 Q&A
- 별도의 외부 그래프 DB 서버 설치 없이 파이썬 메모리 상에서 지식 그래프를 생성하고, LangChain으로 노드와 관계를 조회


In [1]:
# 1. 필요 패키지 설치
!pip install -q langchain-community langchain-google-genai networkx matplotlib

In [2]:
# 2. Gemini API 키 설정

import os
from google.colab import userdata

# Colab 비밀 키에서 GOOGLE_API_KEY를 가져와 환경 변수로 설정합니다.
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception as e:
    print("경고: Colab Secrets에서 GOOGLE_API_KEY를 설정했는지 확인해 주세요.")

In [3]:
model="gemini-2.5-flash"

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 3. LLM 정의
llm = ChatGoogleGenerativeAI(model= model, temperature=0)


In [5]:
from langchain_community.graphs import NetworkxEntityGraph
from langchain_community.graphs.networkx_graph import KnowledgeTriple

# 4. Graph DB (NetworkX) 인스턴스 생성 및 데이터 입력
graph = NetworkxEntityGraph()

# 노드(Entity)와 엣지(Relation) 추가 (KnowledgeTriple 사용)
graph.add_triple(KnowledgeTriple("LangChain", "is_a", "LLM Framework"))
graph.add_triple(KnowledgeTriple("LangChain", "supports", "Graph DB"))
graph.add_triple(KnowledgeTriple("OpenWiki", "built_on", "Deep Agents"))
graph.add_triple(KnowledgeTriple("OpenWiki", "generates", "Markdown Wiki"))
graph.add_triple(KnowledgeTriple("Markdown Wiki", "acts_as", "Knowledge Graph"))

/tmp/ipykernel_8410/1252619513.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import NetworkxEntityGraph


In [6]:
from langchain_community.chains.graph_qa.base import GraphQAChain
# 5. LangChain의 GraphQAChain을 이용해 그래프 질의응답 처리
chain = GraphQAChain.from_llm(llm=llm, graph=graph, verbose=True)

In [7]:
# 6. 질의응답 테스트
response = chain.run("OpenWiki는 무엇 기반으로 구축되었고 무엇을 생성하나요?")
print("\n[답변]:", response)

/tmp/ipykernel_8410/520340534.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = chain.run("OpenWiki는 무엇 기반으로 구축되었고 무엇을 생성하나요?")




> Entering new GraphQAChain chain...
Entities Extracted:
OpenWiki
Full Context:
OpenWiki built_on Deep Agents
OpenWiki generates Markdown Wiki

> Finished chain.

[답변]: OpenWiki는 Deep Agents 기반으로 구축되었고 Markdown Wiki를 생성합니다.


### 2. LangChain의 LLMGraphTransformer
- 자연어 텍스트를 입력했을 때 Gemini가 자동으로 노드와 관계를 추출하여 Graph DB 형태의 트립플(Triple) 데이터로 변환


In [8]:
# 1. 패키지 설치
!pip install -q langchain-experimental langchain-community

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

# # 2. API 키 설정
# os.environ["GEMINI_API_KEY"] = "your-gemini-api-key-here"

# # 3. LLM 및 그래프 변환기 정의
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)
llm_transformer = LLMGraphTransformer(llm=llm)

# 4. 분석할 샘플 문서 준비
text = """
OpenWiki는 코드베이스에 대한 마크다운 위키를 생성하는 CLI 도구입니다.
OpenWiki는 Deep Agents를 기반으로 만들어졌으며, LangSmith 추적을 지원합니다.
에이전트는 위키를 먼저 읽고 코드를 분석하여 토큰 비용을 절약합니다.
"""
docs = [Document(page_content=text)]

# 5. 텍스트에서 그래프 구조(노드 & 관계) 자동 추출
graph_documents = llm_transformer.convert_to_graph_documents(docs)

# 6. 추출된 노드와 엣지(관계) 확인
print("=== 추출된 노드(Nodes) ===")
for node in graph_documents[0].nodes:
    print(f"Node ID: {node.id}, Type: {node.type}")

print("\n=== 추출된 관계(Relationships) ===")
for edge in graph_documents[0].relationships:
    print(f"[{edge.source.id}] --({edge.type})--> [{edge.target.id}]")

/tmp/ipykernel_8410/2331589735.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


=== 추출된 노드(Nodes) ===
Node ID: Openwiki, Type: Tool
Node ID: Cli Tool, Type: Tool type
Node ID: Markdown Wiki, Type: Document type
Node ID: Codebase, Type: Concept
Node ID: Deep Agents, Type: Technology
Node ID: Langsmith, Type: Technology
Node ID: Agent, Type: Component
Node ID: Token Cost, Type: Concept

=== 추출된 관계(Relationships) ===
[Openwiki] --(IS_A)--> [Cli Tool]
[Openwiki] --(CREATES)--> [Markdown Wiki]
[Markdown Wiki] --(FOR)--> [Codebase]
[Openwiki] --(BASED_ON)--> [Deep Agents]
[Openwiki] --(SUPPORTS)--> [Langsmith]
[Agent] --(READS)--> [Markdown Wiki]
[Agent] --(ANALYZES)--> [Codebase]
[Agent] --(SAVES)--> [Token Cost]


## 3. Neo4j 무료 클라우드(AuraDB) 연결
- 실제 시각화가 가능한 Neo4j 데이터베이스에 연결하고 싶다면 AuraDB(무료 티어) 인스턴스를 생성한 후 langchain_community.graphs.Neo4jGraph를 통해 URI 및 비밀번호만 추가로 입력

In [10]:
!pip install -q neo4j

AuraDB 접속 정보 발급 방법: [Neo4j Aura Console](https://console.neo4j.io)에 접속하여 회원가입 후 Free Instance를 생성하면 주는
- Connection URI,
- Username(neo4j),
- Password를 입력